# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## My Baseline Rule

The goal of this baseline is to identify pages that already receive a good number of impressions but have a lower-than-expected click-through rate (CTR). These pages are likely ranking reasonably well but are not attracting enough clicks, making them good candidates for title or meta description improvements.

### Rule

If a page:

- has Position ≤ 10,
- has more than 500 impressions,
- and has CTR below the expected threshold,

then it receives a higher action score.

Pages with larger impression counts receive higher priority because improving their CTR could generate more additional traffic.

### Reason Code

| Reason Code | Meaning | Action |
|-------------|---------|--------|
| LOW_CTR_HIGH_POSITION | High impressions but CTR is lower than expected | Improve title and meta description |

In [4]:
# -----------------------------
# Build the baseline rule
# -----------------------------

# Rule conditions
high_impressions = (df["gsc_impressions"] >= 500).astype(int)
good_position = (df["gsc_avg_position"] <= 10).astype(int)
low_ctr = (df["gsc_ctr"] < 0.05).astype(int)

# Transparent baseline score
df["score"] = (
    high_impressions
    * good_position
    * low_ctr
    * df["gsc_impressions"]
)

# Reason codes
df["reason_code"] = np.where(
    df["score"] > 0,
    "LOW_CTR_HIGH_POSITION",
    "NO_ACTION"
)

# Suggested action
df["action"] = np.where(
    df["score"] > 0,
    "Improve title/meta description",
    "No action"
)

print("Rule successfully created.\n")

display(
    df[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_ctr",
            "gsc_avg_position",
            "score",
            "reason_code",
            "action",
        ]
    ].head(10)
)

Rule successfully created.



,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position,score,reason_code,action
0,20,0,0.000000,3.350000,0,NO_ACTION,No action
1,1,0,0.000000,0.000000,0,NO_ACTION,No action
2,125,1,0.008000,4.928000,0,NO_ACTION,No action
3,7,0,0.000000,4.000000,0,NO_ACTION,No action
4,11,0,0.000000,2.272727,0,NO_ACTION,No action
5,239,1,0.004184,7.347280,0,NO_ACTION,No action
6,191,0,0.000000,7.832461,0,NO_ACTION,No action
7,55,0,0.000000,3.272727,0,NO_ACTION,No action
8,77,0,0.000000,5.636364,0,NO_ACTION,No action
9,2,0,0.000000,4.500000,0,NO_ACTION,No action


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# -----------------------------
# Build the ranked action queue
# -----------------------------

# Keep only pages that require action
queue = (
    df[df["score"] > 0]
    .sort_values(
        by="score",
        ascending=False
    )
    .reset_index(drop=True)
)

# Show rank
queue["rank"] = queue.index + 1

# Select useful output columns
baseline_queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_ctr",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action",
    ]
]

# Display top recommendations
print("Top ranked pages:")
display(baseline_queue.head(20))

# Save ranked queue
output_path = "baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)


print(f"CSV successfully written to: {output_path}")

Top ranked pages:


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position,score,reason_code,action
0,1,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-04,39003,2,0.000051,2.764916,39003,LOW_CTR_HIGH_POSITION,Improve title/meta description
1,2,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.000000,8.613948,37368,LOW_CTR_HIGH_POSITION,Improve title/meta description
2,3,client_62f4a7e64f5e0096,content_60b99970e55b1ac5,2026-03-04,16833,2,0.000119,3.914157,16833,LOW_CTR_HIGH_POSITION,Improve title/meta description
3,4,client_62f4a7e64f5e0096,content_ed50f7f4237a3d02,2026-03-04,16369,6,0.000367,2.519458,16369,LOW_CTR_HIGH_POSITION,Improve title/meta description
4,5,client_62f4a7e64f5e0096,content_465d20bc90052258,2026-03-04,14367,12,0.000835,3.277163,14367,LOW_CTR_HIGH_POSITION,Improve title/meta description
5,6,client_62f4a7e64f5e0096,content_0c5606abaaab3178,2026-03-04,13827,0,0.000000,4.112533,13827,LOW_CTR_HIGH_POSITION,Improve title/meta description
6,7,client_e547b89c05043229,content_ec2e0346994fb5a5,2026-03-01,12303,52,0.004227,1.645615,12303,LOW_CTR_HIGH_POSITION,Improve title/meta description
7,8,client_62f4a7e64f5e0096,content_0bca6d9a85a9b408,2026-03-04,11464,0,0.000000,7.485694,11464,LOW_CTR_HIGH_POSITION,Improve title/meta description
8,9,client_62f4a7e64f5e0096,content_bb2a9972810ddd72,2026-03-04,11213,1,0.000089,3.365469,11213,LOW_CTR_HIGH_POSITION,Improve title/meta description
9,10,client_62f4a7e64f5e0096,content_ff8941941141101f,2026-03-04,10208,2,0.000196,1.722766,10208,LOW_CTR_HIGH_POSITION,Improve title/meta description


CSV successfully written to: baseline_action_score.csv


## 3. Top-20 review

The following review summarizes the highest-ranked pages produced by the baseline rule. Each recommendation includes the proposed action, the reason code, a confidence assessment, and a note describing situations where the recommendation could be incorrect.

In [8]:
# Top 20 recommendations
top20 = baseline_queue.head(20).copy()

# Confidence notes
top20["confidence_note"] = (
    "Medium - Page has high visibility but unusually low CTR."
)

# Possible failure cases
top20["what_would_make_it_wrong"] = (
    "Low CTR may be caused by branded searches, seasonal intent, or user behavior rather than poor titles or metadata."
)

review = top20[
    [
        "rank",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

display(review)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
1,2,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
2,3,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
3,4,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
4,5,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
5,6,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
6,7,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
7,8,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
8,9,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."
9,10,Improve title/meta description,LOW_CTR_HIGH_POSITION,Medium - Page has high visibility but unusuall...,"Low CTR may be caused by branded searches, sea..."


## 4. Weak picks + leakage check

### Weak Picks

Although the baseline rule is transparent and easy to interpret, it is not perfect. Some recommendations may be false positives.

Potential weak picks include:

- Pages with low CTR due to **branded searches**, where users intentionally choose another result.
- **Seasonal content**, where search behavior changes during different times of the year.
- Pages with **special search features** (such as featured snippets or knowledge panels) that naturally reduce CTR.
- Pages with high impressions but **search intent that cannot be improved** by changing the title or meta description.

These examples highlight why the baseline should be considered a decision-support tool rather than a final decision maker.

---

### Leakage Check

The baseline score was built using only features that are available at prediction time:

- Search impressions
- Search clicks
- Click-through rate (CTR)
- Average search position

No future information, manually assigned product flags, or target labels were used when calculating the score.

Therefore, no data leakage was intentionally introduced into this baseline.

In [9]:
# -----------------------------
# Weak Picks Review
# -----------------------------

print("Sample weak picks for manual inspection:")

display(
    baseline_queue[
        [
            "rank",
            "gsc_impressions",
            "gsc_ctr",
            "gsc_avg_position",
            "reason_code",
            "action"
        ]
    ].tail(10)
)

print("\nLeakage Check")

print("✓ No future performance metrics were used.")
print("✓ No manually created product flags were used.")
print("✓ Only current search performance features were used.")
print("✓ The baseline is fully rule-based and transparent.")

Sample weak picks for manual inspection:


,rank,gsc_impressions,gsc_ctr,gsc_avg_position,reason_code,action
1597,1598,501,0.003992,2.722555,LOW_CTR_HIGH_POSITION,Improve title/meta description
1598,1599,501,0.007984,1.596806,LOW_CTR_HIGH_POSITION,Improve title/meta description
1599,1600,500,0.000000,1.322000,LOW_CTR_HIGH_POSITION,Improve title/meta description
1600,1601,500,0.002000,4.164000,LOW_CTR_HIGH_POSITION,Improve title/meta description
1601,1602,500,0.004000,5.398000,LOW_CTR_HIGH_POSITION,Improve title/meta description
1602,1603,500,0.010000,8.840000,LOW_CTR_HIGH_POSITION,Improve title/meta description
1603,1604,500,0.004000,3.204000,LOW_CTR_HIGH_POSITION,Improve title/meta description
1604,1605,500,0.006000,2.032000,LOW_CTR_HIGH_POSITION,Improve title/meta description
1605,1606,500,0.008000,0.922000,LOW_CTR_HIGH_POSITION,Improve title/meta description
1606,1607,500,0.010000,4.646000,LOW_CTR_HIGH_POSITION,Improve title/meta description



Leakage Check
✓ No future performance metrics were used.
✓ No manually created product flags were used.
✓ Only current search performance features were used.
✓ The baseline is fully rule-based and transparent.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.